# Descriptive statistics

Our analysis is based on the gendered dimension of both journalists and athletes. 
Therefore, we provide descriptive statistics on the distribution of male and female journalists and athletes, as well as a cross-tabulated breakdown.

In [1]:
import pandas as pd
import os

# change the path based on the focused sport
data_path = "PESSD/biath/"

df = pd.DataFrame()
for file in os.listdir(data_path) : 
    temp = pd.read_csv(data_path+file)
    df = pd.concat([df,temp])

Now, let's join metadata in order to obtain gender of athletes.

In [2]:
df["ID"] = df["audio_file"].str.replace("_wav.wav","")
df = df.merge(pd.read_csv("metadata.csv", sep=","), on="ID")
df

,start,stop,text,main_speaker,main_g,audio_file,ID,g_ath
0,0.00,12.20,"Deux Français, on va prier en ce dimanche pour...",SPEAKER_02,male,BAH_wav.wav,BAH,H
1,12.20,28.80,"Ouais, ça... Pile au moment où Bouton lance un...",SPEAKER_03,male,BAH_wav.wav,BAH,H
2,28.80,31.80,"Et surtout dans cette zone, oui.",SPEAKER_08,male,BAH_wav.wav,BAH,H
3,31.80,38.80,"Le travail des techniciens, on voit la glisse ...",SPEAKER_02,male,BAH_wav.wav,BAH,H
4,38.80,44.80,On va arriver au deuxième tir. J'ai l'impressi...,SPEAKER_08,female,BAH_wav.wav,BAH,H
...,...,...,...,...,...,...,...,...
3179,2597.08,2604.40,"Vobornikova, ils attendaient une médaille depu...",SPEAKER_02,male,BMF_wav.wav,BMF,F
3180,2604.56,2614.76,Qui est ici? On avait cru pendant quelques ann...,SPEAKER_01,male,BMF_wav.wav,BMF,F
3181,2614.92,2656.40,qui vient chercher cette médaille de bronze. J...,SPEAKER_02,male,BMF_wav.wav,BMF,F
3182,2656.40,2661.76,Et Ossiane qui parle tout le temps de manger. ...,SPEAKER_00,female,BMF_wav.wav,BMF,F


## Descriptive statistics

First, we want to see what is the total duration of the live commentary and how it is distributed among men and women journalists.

In [3]:
# Duration of each segment
df['duration'] = df['stop'] - df['start']

# Total duration
print("Total:")
print(df['duration'].sum() / 60) 

# Journalists
print("Journalists:")
print(df.groupby('main_g')['duration'].sum() / 60) 


Total:
648.3223333333334
Journalists:
main_g
female      104.878000
male        526.731000
music         9.795333
noEnergy      1.748667
noise         5.169333
Name: duration, dtype: float64


Now we want to look at the number of different events based on the gender of athletes (mixt events also are included).

In [4]:
print(df.groupby('g_ath')['duration'].sum() / 60)


g_ath
F    239.291667
H    344.645667
M     64.385000
Name: duration, dtype: float64


Finally, we make a cross tabulation of both journalists' and athletes' gender.

In [5]:

# cross
print("\n Duration (minutes): ")
cross = df.pivot_table(
    values='duration',
    index='main_g',
    columns='g_ath',
    aggfunc='sum'
) / 60
print(cross.round(1))


 Duration (minutes): 
g_ath         F      H     M
main_g                      
female     41.9   51.2  11.8
male      191.8  283.1  51.8
music       2.8    7.0   NaN
noEnergy    NaN    1.3   0.4
noise       2.8    2.1   0.3


Now, we do similar descriptive statistics by replacing time by number of sentences as a unit of analysis.

In [6]:
# dividing texte into sentences
df["text"] = df["text"].apply(lambda x : x.split(". "))
df = df.explode("text")


By athletes' gender:

In [7]:
g_ath_counts = df.groupby('g_ath').size()
print(g_ath_counts)
print((g_ath_counts / g_ath_counts.sum() * 100).round(1))

g_ath
F    3292
H    4830
M     907
dtype: int64
g_ath
F    36.5
H    53.5
M    10.0
dtype: float64


By speakers' gender:

In [33]:
main_g_counts = df.groupby('main_g').size()
print(main_g_counts)
print((main_g_counts / main_g_counts.sum() * 100).round(1))

main_g
female      3844
male        8210
music        224
noEnergy      59
noise        149
dtype: int64
main_g
female      30.8
male        65.8
music        1.8
noEnergy     0.5
noise        1.2
dtype: float64


Cross tabulation:

In [8]:
df_clean = df.reset_index(drop=True) # doublon index issues
print(pd.crosstab(df_clean['main_g'], df_clean['g_ath']))

g_ath        F     H    M
main_g                   
female     688   805  194
male      2547  3903  707
music       39    82    0
noEnergy     0    19    4
noise       18    21    2


## Sentences: descriptive statistics

Now we compare the average length (in words) and their standard deviation for each type of speaker.

In [9]:
df['text'].str.split().str.len().groupby(df['main_g']).agg(['mean', 'std']).round(1)

,mean,std
main_g,,
female,11.2,10.9
male,12.0,11.5
music,9.3,11.5
noEnergy,6.6,5.6
noise,8.3,10.1


Gender of athletes:

In [45]:
df['text'].str.split().str.len().groupby(df['g_ath']).agg(['mean', 'std']).round(1)

,mean,std
g_ath,,
F,11.6,10.2
H,12.1,11.1
M,12.1,10.0


And in general:

In [44]:
df['text'].str.split().str.len().agg(['mean', 'std']).round(1)

mean    12.0
std     10.7
Name: text, dtype: float64

In [10]:
df['word_count'] = df['text'].str.split().str.len()
print(df['word_count'].describe())
print((df['word_count'] < 5).sum(), "segments de moins de 5 mots")

count    9029.000000
mean       11.812382
std        11.426821
min         1.000000
25%         6.000000
50%         9.000000
75%        14.000000
max       235.000000
Name: word_count, dtype: float64
1418 segments de moins de 5 mots


In [11]:
print((df['word_count'] < 5).sum())
print((df['word_count'] == 0).sum())

1418
0
